In [2]:
import pandas as pd
from google.colab import drive
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import RandomizedSearchCV, LeaveOneGroupOut

# Mount Google Drive
drive.mount('/content/drive')

# Define the path to the dataset
dataset_path = '/content/drive/MyDrive/Colab/windowed.csv'

# Load the dataset
try:
    df = pd.read_csv(dataset_path)
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print(f"Error: The file '{dataset_path}' was not found.")
    df = None

if df is not None:
    # Assign the loaded DataFrame to windowed_df
    windowed_df = df.copy()

    # Assuming 'class' is target and 'user' is the grouping column
    if 'class' in windowed_df.columns and 'user' in windowed_df.columns:
        X_windowed = windowed_df.drop(columns=['class', 'user'])
        y_windowed = windowed_df['class']
        groups = windowed_df['user']
        print("\nFeatures (X_windowed) and target (y_windowed) defined.")
    else:
        print("Error: 'class' or 'user' column not found.")
        X_windowed = None
        y_windowed = None

# Proceed with Manual Nested LOSO Cross-Validation
if df is not None and X_windowed is not None and y_windowed is not None:
    print("\nStarting Nested LOSO Cross-Validation with Randomized Search...")

    # Hyperparameter space
    param_dist_rf = {
        'n_estimators': [100, 200, 300, 500],
        'max_depth': [None, 10, 20, 30, 50],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 'log2'],
        'bootstrap': [True, False]
    }

    # Initialize LOSO and storage for scores
    loso = LeaveOneGroupOut()
    inner_f1_scores = []
    outer_f1_scores = []

    print(f"{'Fold':<8} | {'Inner Best F1':<15} | {'Outer Test F1':<15}")
    print("-" * 45)

    # Manual loop for Nested CV
    # We iterate through each subject as a test set
    for i, (train_index, test_index) in enumerate(loso.split(X_windowed, y_windowed, groups)):
        X_train, X_test = X_windowed.iloc[train_index], X_windowed.iloc[test_index]
        y_train, y_test = y_windowed.iloc[train_index], y_windowed.iloc[test_index]

        # Inner Loop: Hyperparameter Tuning
        # inner_cv=3 means it does a 3-fold CV on the training subjects
        rf_classifier = RandomForestClassifier(random_state=42)
        random_search_rf = RandomizedSearchCV(
            estimator=rf_classifier,
            param_distributions=param_dist_rf,
            n_iter=1,
            cv=3,
            scoring='f1_weighted',
            n_jobs=-1,
            verbose=0,
            random_state=42
        )

        # Fit the search (Inner CV happens here)
        random_search_rf.fit(X_train, y_train)

        # Get the best inner score (mean F1 across inner folds)
        best_inner_f1 = random_search_rf.best_score_
        inner_f1_scores.append(best_inner_f1)

        # Outer Loop: Evaluate the best model on the held-out user
        best_model = random_search_rf.best_estimator_
        y_pred = best_model.predict(X_test)
        outer_f1 = f1_score(y_test, y_pred, average='weighted')
        outer_f1_scores.append(outer_f1)

        print(f"Fold {i+1:<4} | {best_inner_f1:<15.4f} | {outer_f1:<15.4f}")

    # Summary Results
    print("\n--- Final Nested LOSO Results ---")
    print(f"Mean Inner F1 (Tuning): {np.mean(inner_f1_scores):.4f}")
    print(f"Mean Outer F1 (Generalization): {np.mean(outer_f1_scores):.4f}")
    print(f"Standard Deviation (Outer): {np.std(outer_f1_scores):.4f}")

else:
    print("Skipping cross-validation due to missing data.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset loaded successfully.

Features (X_windowed) and target (y_windowed) defined.

Starting Nested LOSO Cross-Validation with Randomized Search...
Fold     | Inner Best F1   | Outer Test F1  
---------------------------------------------
Fold 1    | 0.9135          | 0.8947         
Fold 2    | 0.8820          | 0.9161         
Fold 3    | 0.9468          | 0.7049         
Fold 4    | 0.9755          | 0.8736         

--- Final Nested LOSO Results ---
Mean Inner F1 (Tuning): 0.9294
Mean Outer F1 (Generalization): 0.8473
Standard Deviation (Outer): 0.0836
